In [1]:
# Deep Learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, random_split
from torchsummary import summary

# Audio processing
import torchaudio
import torchaudio.transforms as T
import librosa

# Pre-trained image models
# import timm

# Play the audio in Jupyter notebook
from IPython.display import Audio
import pandas as pd
import os
import numpy as np

from scripts.dataset import AudioDataset
from models.cnn_tutorial import CNNNetworkTutorial

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(DEVICE)

cuda


In [2]:
class_mapping = [
    "positive",
    "negative"
]

In [3]:
def train_single_epoch(model, dataloader, criterion, optimizer):
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model.forward(inputs)
        #outputs = model(inputs)

        # Compute the loss and its gradients
        loss = criterion(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

    return loss


def train(model, dataloader, criterion, optimizer, epochs, train_set):
    for epoch in range(epochs):
        loss = train_single_epoch(model, dataloader, criterion, optimizer)
        if epoch % (epochs // 1) == 0:
            print(f"iteration {epoch}: loss {loss.item()} | accuracy: {accuracy(model, train_set)}")
            # Get model weights after each iteration
            model_weights = model.state_dict()
            print("Model Weights:", model_weights)


def predict(model, input, target):
    model.eval()
    with torch.no_grad():
        predictions = model(input)
        # Tensor (1, 10) -> [ [0.1, 0.01, ..., 0.6] ]
        predicted_index = predictions[0].argmax(0)
        predicted = class_mapping[predicted_index]
        expected = class_mapping[target]
    return predicted, expected

def accuracy(model, dataset):
    correct = 0
    total_samples = len(dataset)
    for idx in dataset.indices:
        input = dataset[dataset.indices.index(idx)][0].to(DEVICE)
        target = dataset[dataset.indices.index(idx)][1].argmax(0)
        input.unsqueeze_(0)

        predicted, expected = predict(model, input, target)
        
        if predicted == expected:
            correct += 1
    
    return (correct / total_samples) * 100

In [4]:
AUDIO_DIR = "audios/labeled/"

dict_genres = {'positive': 0, 'negative': 1}


reverse_map = {v: k for k, v in dict_genres.items()}
print(reverse_map)

{0: 'positive', 1: 'negative'}


In [5]:
data = []

for label in dict_genres.keys():
    for folder in os.listdir(AUDIO_DIR + label):
        for file in os.listdir(AUDIO_DIR + label + "/" + folder):
            file_path = AUDIO_DIR + label + "/" + folder + "/" + file
            positive = int(label == "positive")
            negative = int(label == "negative")
            data.append((file_path, positive,
                        negative))

file_path, positive, negative = zip(*data)
df = pd.DataFrame({"file_path": file_path, "positive": positive, "negative": negative})

print(df.head(5))

                                 file_path  positive  negative
0  audios/labeled/positive/1001/100100.wav         1         0
1  audios/labeled/positive/1002/100200.wav         1         0
2  audios/labeled/positive/1003/100300.wav         1         0
3  audios/labeled/positive/1003/100301.wav         1         0
4  audios/labeled/positive/1003/100302.wav         1         0


In [8]:
BATCH_SIZE = 1
EPOCHS = 10
LEARNING_RATE = 0.00005
SAVE = False
TRAIN_RATIO = 0.8
TEST_RATIO = 1 - TRAIN_RATIO

dataset = AudioDataset(df)

# Calculate the lengths of training and testing subsets
train_length = int(len(dataset) * TRAIN_RATIO)
test_length = len(dataset) - train_length

# Use random_split to split the dataset
train_set, validation_set = random_split(dataset, [train_length, test_length])

# Create data loaders for training and testing
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
#validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False)

# construct model and assign it to device
model = CNNNetworkTutorial().to(DEVICE)
#summary(model.cuda(), (1, 64, 44))

# initialise loss funtion + optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                                lr=LEARNING_RATE)

# train model
train(model, train_loader, criterion, optimizer, EPOCHS, train_set)

print("Train Set Accuracy: ", accuracy(model, train_set),"%")
print("Validation Set Accuracy: ", accuracy(model, validation_set),"%")

# save model
if SAVE:
    torch.save(model.state_dict(), "feedforwardnet.pth")
    print("Trained feed forward net saved at feedforwardnet.pth")

iteration 0: loss 0.3132617473602295 | accuracy: 54.285714285714285
Model Weights: OrderedDict([('conv1.0.weight', tensor([[[[-0.0615,  0.2771,  0.1988],
          [ 0.1822,  0.2815, -0.2448],
          [ 0.2258, -0.0855, -0.3274]]],


        [[[-0.1900,  0.1446, -0.3111],
          [ 0.1491, -0.1994, -0.2231],
          [ 0.0965, -0.1408, -0.2266]]],


        [[[-0.1787,  0.2582, -0.1094],
          [ 0.2758, -0.2614,  0.2385],
          [ 0.1870,  0.1178,  0.0270]]],


        [[[ 0.2544, -0.2202,  0.1561],
          [-0.0432,  0.2628, -0.0516],
          [-0.3057, -0.0522, -0.1372]]],


        [[[-0.1949, -0.3193, -0.1590],
          [ 0.2774, -0.2293,  0.2297],
          [-0.1831, -0.1866,  0.2659]]],


        [[[-0.3055, -0.3259,  0.1831],
          [-0.1776, -0.0618,  0.0024],
          [-0.1872, -0.2198, -0.0752]]],


        [[[-0.3209, -0.0216, -0.1999],
          [-0.2476,  0.3227,  0.0860],
          [ 0.2089,  0.0574,  0.1837]]],


        [[[-0.2885,  0.1214,  0.0664],

In [7]:
input, target = dataset[40][0].to(DEVICE), dataset[40][1].argmax(0)
input.unsqueeze_(0)


predicted, expected = predict(model, input, target)
print(f"Predicted: '{predicted}', expected: '{expected}'")

Predicted: 'positive', expected: 'negative'
